# NB-20 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-20**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-20.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This constructs a directed relation from retained gross flows and separates symmetric geometry from direction-dependent drift. The construction is descriptive, not a causal network.

- [Volume III, Chapter 5 — Temporal graph analysis](https://sherandoresearch.github.io/power-markets-series/vol3_spectral_grids.html#c5)
- [Volume IV, Chapter 11 — Modal dynamics, closed loop](https://sherandoresearch.github.io/power-markets-series/vol4_testbed_atlas.html#chiv-11)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-20', require_key=True)
print(setup_status)


# NB-20 — Directed affinity: geometry versus drift
Companion to Volume III §5 and Volume IV §11; prerequisites NB-04 and NB-19.
Reuse the retained Canonical E dispatch, injections and grid. No new injections,
random flows or market observations are generated. This is a controlled
gross-flow relation, not a causal network or a fitted manifold.

In [ ]:
NOTEBOOK_ID='NB-20'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return {k:response[k] for k in ('result','diagnostics')}
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

print('Local retained injection inspection:',inputs['raw_sample_inspection'])
print('Operator provenance:',inputs['operator_scope'])
windows,affinities,operators=[],[],[]
def generators(name,mean,policy,parent,control='original'):
    a=np.asarray(mean['values'],float);mask=np.asarray(mean['support_mask'],bool)
    if control=='transpose':a=a.T;mask=mask.T
    elif control=='symmetrized':a=(a+a.T)/2
    assert control in ('original','transpose','symmetrized')
    req=dict(inputs=dict(directed_affinity=dict(node_ids=mean['node_ids'],values=a.tolist(),support_mask=mask.tolist(),
        row_convention='origin_row_destination_column',affinity_unit='MW',
        observation_window=dict(kind='indexed_interval',support_id=policy['support_id'],start_index=policy['start_index'],
            end_index_exclusive=policy['end_index_exclusive'],sample_interval_seconds=3600,utc_calendar_inferred=False),
        support_policy='unobserved_excluded',self_loop_policy='reject',
        source=dict(reference_id='NB-20:'+parent,artifact_sha256=responses[parent]['receipt']['result_sha256'],
            construction='Retained controlled network calculation; mean positive branch flow; '+control+' algebraic control')),
        generator_parameters=dict(theta=1,bandwidth=1,bandwidth_unit='dimensionless',normalization='symmetric_source_density_power',
            zero_degree_policy='reject',kernel_convention='source_zero_row_sum')),
        parameters=dict(execution_profile='declared_indexed_affinity'),context=dict(input_origin='Retained indexed source; no inferred UTC dates or causality'))
    output=execute(name,'directed_generators',req,[parent])['result']['directed_generator_bundle']
    return np.asarray(output['geometry']),np.asarray(output['drift'])
for number,window in enumerate(inputs['windows']):
    rows,ids,indices=[],[],[]
    for parent in window['network_parents']:
        req=inputs['requests'][parent]
        response=execute(parent,'dc_network_response',req)
        flow=response['result']['flow_field'];declared=req['inputs']['balanced_injections']
        assert flow['case_ids']==declared['case_ids'] and flow['sample_indices']==declared['sample_indices']
        assert flow['constraint_ids']==req['inputs']['electrical_network']['constraint_ids']
        assert response['diagnostics']['checked_cases']==len(flow['case_ids'])
        rows.extend(flow['values']);ids.extend(flow['case_ids']);indices.extend(flow['sample_indices'])
    template=copy.deepcopy(window['affinity_template']);record=template['border_flow_records'];policy=template['affinity_construction_policy']
    assert ids==record['case_ids'] and indices==record['sample_indices']
    record['values']=rows
    req=dict(inputs=template,parameters=dict(execution_profile='retained_indexed_flow_window'),
        context=dict(input_origin='Complete original eligibility roster and every corresponding private DC panel output'))
    base=f'window{number}-affinity'
    response=execute(base,'border_flow_affinity',req,window['network_parents'])
    mean=response['result']['directed_affinity'];support=response['result']['support_ledger']
    assert support['eligibility']==record['eligibility'] and support['selected_case_ids']==ids
    assert support['included_hours']+support['excluded_hours']==4380 and not support['missing_values_estimated']
    g,d=generators(f'window{number}-generators',mean,policy,base)
    gt,dt=generators(f'window{number}-transpose',mean,policy,base,'transpose')
    reversed_req=copy.deepcopy(req)
    reversed_req['inputs']['border_flow_records']['endpoints']=[pair[::-1] for pair in record['endpoints']]
    reversed_req['inputs']['border_flow_records']['values']=(-np.asarray(rows)).tolist()
    reversed_result=execute(f'window{number}-reversed-coordinates','border_flow_affinity',reversed_req,window['network_parents'])['result']['directed_affinity']
    network=inputs['requests'][window['network_parents'][0]]['inputs']['electrical_network']
    context=dict(model_context_id=network['model_context_id'],topology_state_id=network['topology_state_id'],
        valid_interval=dict(kind='fixture_support',id=policy['support_id']+':'+str(policy['start_index'])+':'+str(policy['end_index_exclusive'])),
        weight_unit='normalized_susceptance',weight_meaning='Original controlled weights reconstructed from retained operators',
        flow_unit='MW',power_normalization='source_normalized_weights_no_physical_MVA_base_declared',
        edge_orientation='positive_from_first_endpoint_to_second')
    physical=dict(inputs=dict(electrical_network=dict(node_ids=network['node_ids'],edge_ids=network['constraint_ids'],
        endpoints=record['endpoints'],weights=network['susceptance'],active_mask=network['active_mask'],island_policy='reject',context=context),
        edge_flow_vector=dict(edge_ids=network['constraint_ids'],values=response['diagnostics']['mean_signed_flow'],context=context)),
        parameters=dict(execution_profile='declared_normalized_network'),context=dict(input_origin='Mean signed flow from complete declared retained window; no physical MVA base inferred'))
    hodge=execute(f'window{number}-hodge','weighted_hodge',physical,[base])['result']['hodge_decomposition']
    a=np.asarray(mean['values']);reversed_a=np.asarray(reversed_result['values'])
    windows.append(dict(start_hour=policy['start_index'],end_hour=policy['end_index_exclusive'],
        included_hours=support['included_hours'],excluded_hours=support['excluded_hours'],
        geometry_transpose_error=float(np.linalg.norm(g-gt)),drift_transpose_error=float(np.linalg.norm(d+dt)),
        orientation_error=float(np.linalg.norm(a-reversed_a)),
        weighted_hodge_relative_residual=float(np.sqrt(hodge['cycle_energy']/hodge['input_energy'])),
        meanflow=response['diagnostics']['mean_signed_flow']))
    affinities.append(a);operators.append((g,d))
changes={kind:float(np.linalg.norm(operators[0][i]-operators[1][i])/np.linalg.norm(operators[0][i])) for i,kind in enumerate(('geometry','drift'))}
old=inputs['original_saved_summary']
for w in windows:print({k:v for k,v in w.items() if k!='meanflow'})
print('Affinities: mean positive directed branch MW, square 26-bus carrier; diagonal zero.')

## Define the relation before analyzing its direction
For a declared edge i→j and physical flow f, aᵢⱼ=mean(max(f,0)) and
aⱼᵢ=mean(max(−f,0)). Gross flow in both directions can be positive even when
mean net flow is small. Reversing the edge coordinate and its signed flow
together leaves this bus-to-bus affinity unchanged. Transposing the affinity
instead reverses the physical directional relation.

Use calendar-hour windows [0,4380) and [4380,8760), excluding outage and
shedding hours so the same intact passive operator applies. Aggregate shedding
lacks bus locations; none are invented. The resulting windows have different
support counts, and the comparison is conditional on this exclusion policy.
No lead-lag statistic or forecast direction is implied.

In [ ]:
a=affinities[0];geometry,drift=operators[0]
from matplotlib.colors import LinearSegmentedColormap
cmap=LinearSegmentedColormap.from_list('nord_diverging',['#5E81AC','#ECEFF4','#BF616A'])
fig,axes=plt.subplots(1,3,figsize=(13,4))
for ax,m,title in zip(axes,[a,geometry,drift],['Gross-flow affinity (MW)','Geometry generator','Drift generator']):
    limit=np.abs(m).max();im=ax.imshow(m,cmap=cmap,vmin=-limit,vmax=limit);fig.colorbar(im,ax=ax,fraction=.046);ax.set(title=title,xlabel='Bus column',ylabel='Bus row')
fig.tight_layout();plt.show()

## Shared normalization and recentering
Define qᵢ=(Σⱼaᵢⱼ+Σⱼaⱼᵢ)/2 and bᵢⱼ=aᵢⱼ/(qᵢqⱼ)^θ.
With S=(B+Bᵀ)/2, A=(B−Bᵀ)/2 and dᵢ=ΣⱼSᵢⱼ, the implemented operators are
G=D⁻¹(S−diag(S1))/h² and R=D⁻¹(A−diag(A1))/h.
Here θ=1 and h=1 are declared normalization parameters, not a fitted time unit.
Both row sums vanish. D G is symmetric; R need not be skew-symmetric after
row normalization and diagonal recentering.

Transposition keeps q,S,d and G unchanged and reverses R exactly. Symmetrizing
removes directional information. These are bounded operator identities, not a
full Finsler embedding, Randers-admissibility proof or manifold-limit test.

In [ ]:
degree=np.asarray(responses['window0-generators']['result']['directed_generator_bundle']['normalized_degree'])
gt=np.asarray(responses['window0-transpose']['result']['directed_generator_bundle']['geometry'])
rt=np.asarray(responses['window0-transpose']['result']['directed_generator_bundle']['drift'])
mean=responses['window0-affinity']['result']['directed_affinity'];policy=inputs['windows'][0]['affinity_template']['affinity_construction_policy']
gs,rs=generators('window0-symmetrized',mean,policy,'window0-affinity','symmetrized')
np.testing.assert_allclose(geometry,gt,atol=1e-12);np.testing.assert_allclose(drift,-rt,atol=1e-12)
np.testing.assert_allclose(geometry,gs,atol=1e-12);np.testing.assert_allclose(rs,0,atol=1e-12)
np.testing.assert_allclose(geometry.sum(axis=1),0,atol=1e-12);np.testing.assert_allclose(drift.sum(axis=1),0,atol=1e-12)
np.testing.assert_allclose(degree[:,None]*geometry,(degree[:,None]*geometry).T,atol=1e-12)
fig,ax=plt.subplots();ax.plot(np.diag(drift),label='Original drift diagonal',color='#88C0D0');ax.plot(np.diag(rt),label='Transpose',color='#B48EAD');ax.plot(np.diag(rs),label='Symmetrized',color='#D08770');ax.set(xlabel='Bus',ylabel='Recentered drift diagonal',title='Reversal and information-loss controls');ax.legend();plt.show()

## Directed gross flow is not a physical circulation
On the fixed passive grid, the mean physical branch flow remains in im(WA).
Project it with Q=WA L⁺ Aᵀ and measure the residual in W⁻¹. The retained
windows have essentially no physical cycle component, while their gross-flow
affinity has nonzero drift. These measure different objects.

Window changes mix dispatch composition and seasonality with the selected
hours. They are descriptive sensitivity, not uncertainty intervals or evidence
of a topology change. The prior 3,000-draw biased-injection experiment below is
replayed only as a saved summary; those draws are not regenerated.

In [ ]:
print('Current window changes:',changes)
print('Saved 3,000-draw summary ONLY:',old)
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(windows[0]['meanflow'],label='First half',color='#88C0D0');axes[0].plot(windows[1]['meanflow'],label='Second half',color='#B48EAD');axes[0].set(xlabel='Canonical edge coordinate',ylabel='Mean signed MW',title='Raw physical flow comparison');axes[0].legend()
axes[1].bar(['Geometry','Drift'],[changes['geometry'],changes['drift']],color=['#88C0D0','#D08770']);axes[1].set(ylabel='Relative Frobenius change',title='Conditional window sensitivity');fig.tight_layout();plt.show()

## What fails the input contract
A signed rectangular PTDF is not a nonnegative square affinity. Negative entries
and zero shared-degree nodes also fail this implementation's contract. Exclusion
or a different isolated-node policy requires an explicit decision. An incidence
orientation is not asymmetric economic information.

Exercise: what would have to change to define a lead-lag price affinity? State
the sampling clock, lag sign, training cutoff and confounding limits before
interpreting any direction.

In [ ]:
failures=[]
base_request=next(row['request'] for row in execution_plan if row['name']=='window0-generators')
for label,value in [('rectangular signed path operator',inputs['rectangular_control']),('negative affinity',(-a).tolist()),('zero shared degree',(a*0).tolist())]:
    req=copy.deepcopy(base_request);req['inputs']['directed_affinity']['values']=value
    reject('directed_generators',req);failures.append(label)
assert len(failures)==3
bad=copy.deepcopy(inputs['requests'][inputs['windows'][0]['network_parents'][0]])
bad['inputs']['balanced_injections']['sample_indices'].reverse();reject('dc_network_response',bad)
bad=copy.deepcopy(next(row['request'] for row in execution_plan if row['name']=='window0-affinity'))
bad['inputs']['border_flow_records']['eligibility'].pop();reject('border_flow_affinity',bad)

In [ ]:
checked=dict(windows=windows,window_changes=changes,negative_controls=failures,new_random_data=False,
    source_exact_operator_bytes_claimed=False,scientific_acceptance=False)
(session.output/'consumer-summary.json').write_text(json.dumps(checked,indent=2,allow_nan=False)+'\n')
summary=session.finish(checked)
print(summary)